# Week 15: Calling the Real Deployed Container

Same logic as `call_deployed_api.py`. Unlike Week 14's `call_api.py` (which starts an in-process `uvicorn` server in a background thread), this notebook assumes the API is already running as a real, separate process in a real Docker container — the actual "deployment" this week is about. Start it first, from a terminal, in the repo root:

```bash
docker compose up -d --build
```

No `LLM_API_KEY` is needed for `/health` or `/search` — the container's startup lifespan indexes `data/sample/passages.json` automatically, so `/search` has real data to query without running Week 9's `build_passage_index.py` first. `/ask` needs `LLM_API_KEY`/`LLM_MODEL` in the `.env` file `docker-compose.yml` passes into the container — without it, `/ask` returns `500`, which is an expected, documented outcome here, not a bug.

In [ ]:
import sys

import httpx

BASE_URL = "http://localhost:8000"

try:
    httpx.get(f"{BASE_URL}/health", timeout=5.0).raise_for_status()
    print("Server is up.")
except httpx.HTTPError:
    print(f"Could not reach {BASE_URL}. Run `docker compose up -d --build` first.", file=sys.stderr)

## GET /health

In [ ]:
with httpx.Client(base_url=BASE_URL, timeout=30.0) as client:
    health = client.get("/health")
print(health.status_code, health.json())

## POST /search — Auto-Indexed Sample Passages, No LLM Call

This works with zero setup beyond `docker compose up` — the container's `lifespan` startup step indexed `data/sample/passages.json` the first time it ran (Week 15 §1.4).

In [ ]:
query = "did the company beat earnings expectations?"

with httpx.Client(base_url=BASE_URL, timeout=30.0) as client:
    search = client.post("/search", json={"query": query, "n_results": 3})

print(search.status_code)
for result in search.json():
    print(f"[{result['distance']:.3f}] ({result['ticker']}) {result['text']}")

## POST /ask — Retrieval + Generation

In [ ]:
with httpx.Client(base_url=BASE_URL, timeout=30.0) as client:
    ask = client.post("/ask", json={"query": query, "n_results": 3})

print(ask.status_code)
if ask.status_code == 200:
    body = ask.json()
    print("answer:", body["answer"])
    print("citations:", body["citations"])
    for source in body["sources"]:
        print(f"  [{source['distance']:.3f}] ({source['ticker']}) {source['text']}")
else:
    # A 500 here usually means LLM_API_KEY/LLM_MODEL aren't set in .env.
    print(ask.text or "(no response body — check LLM_API_KEY/LLM_MODEL are set)")

## Checking the Deployed Container's Logs

Every request above was logged by the container's request-logging middleware, including method, path, status code, and duration — and, for the `/ask` call above (if no `LLM_API_KEY` is set), an `unhandled exception` line with a full traceback, found and fixed in Week 15 §2.3. From a terminal:

```bash
docker compose logs api
```